# NB 02: Spread Analysis (Core Notebook)

Compute pairwise funding rate spreads across all exchange pairs,
analyze persistence (OU half-life), directional consistency,
CEX-CEX vs CEX-DEX vs DEX-DEX comparison, and cointegration.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from scipy import stats
from statsmodels.tsa.stattools import coint
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

from src.config import EXCHANGES, ASSETS, PROCESSED_DIR, FIGURES_DIR
from src.processing.aligner import compute_spreads

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sns.set_style('whitegrid')

# Load aligned data
wide = pd.read_parquet(PROCESSED_DIR / 'funding_rates_aligned.parquet')
print(f'Aligned data: {wide.shape}')
print(f'Columns: {[(e,a) for e,a in wide.columns]}')

## 1. Pairwise Spreads

In [ ]:
# Compute spreads for all exchange pairs x assets
all_spreads = []

for asset in sorted(wide.columns.get_level_values('asset').unique()):
    asset_spreads = compute_spreads(wide, asset)
    if not asset_spreads.empty:
        all_spreads.append(asset_spreads)

spreads_df = pd.concat(all_spreads, ignore_index=True) if all_spreads else pd.DataFrame()
print(f'Total spread observations: {len(spreads_df):,}')

if not spreads_df.empty:
    # Create pair label
    spreads_df['pair'] = spreads_df['exchange_short'] + ' / ' + spreads_df['exchange_long']
    
    # Summary by pair-asset
    spread_summary = spreads_df.groupby(['pair', 'asset'])['spread'].agg(
        count='count',
        mean='mean',
        median='median',
        std='std',
        max='max',
    ).round(6)
    spread_summary

## 2. Ornstein-Uhlenbeck Half-Life Estimation

The OU half-life measures how quickly the spread reverts to its mean.
Shorter half-lives = faster mean reversion = more exploitable.

In [ ]:
def ou_half_life(spread_series: pd.Series) -> float:
    """Estimate Ornstein-Uhlenbeck half-life from a spread series.
    
    Regress: delta_spread = theta * (spread - mean) + noise
    Half-life = -ln(2) / theta
    """
    spread = spread_series.dropna()
    if len(spread) < 20:
        return np.nan
    
    # Demeaned spread
    spread_demean = spread - spread.mean()
    delta = spread_demean.diff().dropna()
    lag = spread_demean.shift(1).dropna()
    
    # Align
    delta = delta.iloc[1:]
    lag = lag.iloc[:len(delta)]
    
    # OLS regression
    X = add_constant(lag)
    model = OLS(delta, X).fit()
    theta = model.params.iloc[1]
    
    if theta >= 0:
        return np.inf  # Not mean-reverting
    
    return -np.log(2) / theta

# Compute OU half-life for each exchange pair
hl_records = []

for asset in sorted(wide.columns.get_level_values('asset').unique()):
    exchanges_for_asset = [e for e, a in wide.columns if a == asset]
    exchanges_for_asset = sorted(set(exchanges_for_asset))
    
    for ex_a, ex_b in combinations(exchanges_for_asset, 2):
        spread = wide[(ex_a, asset)] - wide[(ex_b, asset)]
        spread = spread.dropna()
        
        if len(spread) < 50:
            continue
        
        hl = ou_half_life(spread)
        
        hl_records.append({
            'exchange_a': ex_a,
            'exchange_b': ex_b,
            'asset': asset,
            'half_life_periods': hl,
            'half_life_hours': hl * 8,  # Convert to hours
            'half_life_days': hl * 8 / 24,
            'n_obs': len(spread),
            'pair_type': _pair_type(ex_a, ex_b),
        })

def _pair_type(a, b):
    ta = EXCHANGES[a]['type']
    tb = EXCHANGES[b]['type']
    if ta == 'CEX' and tb == 'CEX': return 'CEX-CEX'
    if ta == 'DEX' and tb == 'DEX': return 'DEX-DEX'
    return 'CEX-DEX'

hl_df = pd.DataFrame(hl_records)
if not hl_df.empty:
    print('OU Half-Life Summary (periods of 8h):')
    print(hl_df.sort_values('half_life_periods').to_string())

## 3. Directional Consistency

How reliably does the spread maintain direction?

In [ ]:
consistency = []

for asset in sorted(wide.columns.get_level_values('asset').unique()):
    exchanges_for_asset = sorted(set(e for e, a in wide.columns if a == asset))
    
    for ex_a, ex_b in combinations(exchanges_for_asset, 2):
        spread = (wide[(ex_a, asset)] - wide[(ex_b, asset)]).dropna()
        if len(spread) < 20:
            continue
        
        # What fraction of time does the spread maintain sign?
        pct_positive = (spread > 0).mean()
        pct_negative = (spread < 0).mean()
        dominant_direction = max(pct_positive, pct_negative)
        
        # Streak analysis: longest consecutive same-sign run
        signs = np.sign(spread)
        changes = signs != signs.shift(1)
        groups = changes.cumsum()
        longest_streak = groups.value_counts().max()
        
        consistency.append({
            'exchange_a': ex_a, 'exchange_b': ex_b, 'asset': asset,
            'pct_a_higher': pct_positive * 100,
            'pct_b_higher': pct_negative * 100,
            'dominant_direction_pct': dominant_direction * 100,
            'longest_streak_periods': longest_streak,
            'mean_spread': spread.mean(),
        })

cons_df = pd.DataFrame(consistency)
if not cons_df.empty:
    cons_df = cons_df.sort_values('dominant_direction_pct', ascending=False)
    print('Directional consistency (higher = more reliable signal):')
    cons_df

## 4. CEX-CEX vs CEX-DEX vs DEX-DEX Comparison

In [ ]:
if not hl_df.empty:
    # Box plot of spreads by pair type
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # 1. Half-life by pair type
    valid_hl = hl_df[hl_df['half_life_periods'] < 100]  # Exclude non-reverting
    if not valid_hl.empty:
        valid_hl.boxplot(column='half_life_days', by='pair_type', ax=axes[0])
        axes[0].set_title('OU Half-Life by Pair Type')
        axes[0].set_ylabel('Half-Life (days)')
        axes[0].set_xlabel('')
    
    # 2. Mean spread by pair type
    if not spreads_df.empty:
        spreads_df['pair_type'] = spreads_df.apply(
            lambda r: _pair_type(r['exchange_short'], r['exchange_long']), axis=1
        )
        pair_means = spreads_df.groupby(['pair_type', 'asset'])['spread'].mean().reset_index()
        pair_means.boxplot(column='spread', by='pair_type', ax=axes[1])
        axes[1].set_title('Mean Spread by Pair Type')
        axes[1].set_ylabel('Spread (8h rate)')
        axes[1].set_xlabel('')
    
    # 3. Directional consistency by pair type
    if not cons_df.empty:
        cons_df['pair_type'] = cons_df.apply(
            lambda r: _pair_type(r['exchange_a'], r['exchange_b']), axis=1
        )
        cons_df.boxplot(column='dominant_direction_pct', by='pair_type', ax=axes[2])
        axes[2].set_title('Directional Consistency by Pair Type')
        axes[2].set_ylabel('Dominant Direction (%)')
        axes[2].set_xlabel('')
    
    plt.suptitle('')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'pair_type_comparison.pdf', bbox_inches='tight')
    plt.show()

## 5. Time-of-Day / Day-of-Week Effects

In [ ]:
if not spreads_df.empty:
    spreads_df['hour'] = spreads_df['timestamp'].dt.hour
    spreads_df['dow'] = spreads_df['timestamp'].dt.day_name()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Hour-of-day
    hourly = spreads_df.groupby('hour')['spread'].mean() * 100
    hourly.plot(kind='bar', ax=ax1)
    ax1.set_title('Mean Spread by Settlement Hour (UTC)')
    ax1.set_ylabel('Mean Spread (%)')
    
    # Day-of-week
    dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    daily = spreads_df.groupby('dow')['spread'].mean().reindex(dow_order) * 100
    daily.plot(kind='bar', ax=ax2)
    ax2.set_title('Mean Spread by Day of Week')
    ax2.set_ylabel('Mean Spread (%)')
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'temporal_effects.pdf', bbox_inches='tight')
    plt.show()

## 6. Cointegration Tests (Engle-Granger)

In [ ]:
coint_results = []

for asset in sorted(wide.columns.get_level_values('asset').unique()):
    exchanges_for_asset = sorted(set(e for e, a in wide.columns if a == asset))
    
    for ex_a, ex_b in combinations(exchanges_for_asset, 2):
        series_a = wide[(ex_a, asset)].dropna()
        series_b = wide[(ex_b, asset)].dropna()
        
        # Align
        common = series_a.index.intersection(series_b.index)
        if len(common) < 50:
            continue
        
        sa = series_a.loc[common]
        sb = series_b.loc[common]
        
        # Engle-Granger test
        score, pvalue, _ = coint(sa, sb)
        
        coint_results.append({
            'exchange_a': ex_a, 'exchange_b': ex_b, 'asset': asset,
            'coint_stat': score, 'coint_p': pvalue,
            'cointegrated': pvalue < 0.05,
            'n_obs': len(common),
        })

coint_df = pd.DataFrame(coint_results)
if not coint_df.empty:
    print(f'Cointegrated pairs (p < 0.05): {coint_df["cointegrated"].sum()}/{len(coint_df)}')
    coint_df.sort_values('coint_p')

## 7. Summary: Most Promising Pairs

In [ ]:
# Rank pairs by exploitability
# Criteria: large mean spread, fast mean-reversion, high directional consistency

if not hl_df.empty and not cons_df.empty and not spreads_df.empty:
    # Merge metrics
    pair_metrics = hl_df.merge(
        cons_df[['exchange_a', 'exchange_b', 'asset', 'dominant_direction_pct', 'mean_spread']],
        on=['exchange_a', 'exchange_b', 'asset'],
        how='outer'
    )
    
    # Score (higher = more promising)
    # Normalize each metric to 0-1 range
    valid = pair_metrics[pair_metrics['half_life_periods'].notna() & 
                        (pair_metrics['half_life_periods'] < 100)].copy()
    
    if not valid.empty:
        valid['spread_score'] = valid['mean_spread'].rank(pct=True)
        valid['reversion_score'] = 1 - valid['half_life_periods'].rank(pct=True)  # Lower = better
        valid['direction_score'] = valid['dominant_direction_pct'].rank(pct=True)
        valid['composite_score'] = (
            valid['spread_score'] + valid['reversion_score'] + valid['direction_score']
        ) / 3
        
        ranking = valid.sort_values('composite_score', ascending=False)
        print('Top 10 Most Promising Exchange Pairs:')
        display_cols = ['exchange_a', 'exchange_b', 'asset', 'pair_type',
                       'mean_spread', 'half_life_days', 'dominant_direction_pct',
                       'composite_score']
        ranking[display_cols].head(10)
    else:
        print('Not enough data for pair ranking')
else:
    print('Missing analysis data for ranking')